# AI Cycling Coach Project: Delta Table Setup & ETL Foundations

This notebook establishes a robust Delta Lake foundation for an AI cycling coach, including athlete profile, ride history, weather, and route schemas. It also scaffolds ETL pipelines for app/wearable ingestion and weather enrichment.

## Completed
* Delta schemas created: athletes, rides, weather, routes
* Sample athlete profiles written to Delta
* Modular ETL logic for rides and weather from Strava, Garmin, TrainingPeaks APIs

## Still To Do
* Expand ETL for Garmin, TrainingPeaks, and additional sensors
* Deepen weather/terrain integration (traffic, elevation, more sources)
* Complete route enrichment with actual geo data
* Implement data governance: Unity Catalog permissions, privacy features
* Build ML models for athlete assessment & training plan personalization
* Develop dashboards & reporting for feedback and analytics
* Design web/mobile app interface and API endpoints for athlete and coach interaction
* Add adaptability logic: flexible plans, chat, notifications, event strategy, feedback

---
This notebook is the first module in the full cycling coach pipeline — subsequent notebooks will cover training plan generation, athlete assessment, route planning, live coaching, and analytics.

In [0]:
# Databricks notebook source
# Delta Table Schemas for Cycling Coach Data Model
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, LongType, DateType, TimestampType

# 1. Athlete Profile Schemaathlete_schema = StructType([
    StructField("athlete_id", StringType(), False), # Primary Key
    StructField("name", StringType(), True),
    StructField("birth_date", DateType(), True),
    StructField("gender", StringType(), True),
    StructField("joined_date", DateType(), True),
    StructField("fitness_level", StringType(), True),
    StructField("training_goal", StringType(), True),
    StructField("available_hours_week", IntegerType(), True),
    StructField("injury_history", StringType(), True),
    StructField("notes", StringType(), True)
])

# 2. Ride History Schema
ride_schema = StructType([
    StructField("ride_id", StringType(), False),  # Primary Key
    StructField("athlete_id", StringType(), False),  # FK to athletes
    StructField("ride_date", TimestampType(), True),
    StructField("duration_seconds", IntegerType(), True),
    StructField("distance_km", FloatType(), True),
    StructField("avg_power_watts", FloatType(), True),
    StructField("avg_hr", FloatType(), True),
    StructField("max_power_watts", FloatType(), True),
    StructField("max_hr", FloatType(), True),
    StructField("ride_type", StringType(), True),  # e.g., Interval, Long, Race
    StructField("weather_id", StringType(), True), # FK to weather table
    StructField("route_id", StringType(), True),   # FK to routes table
    StructField("perceived_exertion", IntegerType(), True),
    StructField("notes", StringType(), True)
])

# 3. Weather Schema
weather_schema = StructType([
    StructField("weather_id", StringType(), False),
    StructField("ride_id", StringType(), True), # FK to rides
    StructField("observation_time", TimestampType(), True),
    StructField("temperature_c", FloatType(), True),
    StructField("humidity_pct", FloatType(), True),
    StructField("wind_speed_kmh", FloatType(), True),
    StructField("wind_direction_deg", IntegerType(), True),
    StructField("precipitation_mm", FloatType(), True),
    StructField("visibility_km", FloatType(), True),
    StructField("weather_condition", StringType(), True)
])

# 4. Route Metadata Schema
route_schema = StructType([
    StructField("route_id", StringType(), False),
    StructField("name", StringType(), True),
    StructField("start_location", StringType(), True),
    StructField("end_location", StringType(), True),
    StructField("distance_km", FloatType(), True),
    StructField("elevation_gain_m", IntegerType(), True),
    StructField("terrain_type", StringType(), True),
    StructField("segment_data", StringType(), True)  # JSON/GeoMetadata
])

# Create empty DataFrames as table shells (for schema registration)
athletes_df = spark.createDataFrame([], schema=athlete_schema)
rides_df = spark.createDataFrame([], schema=ride_schema)
weather_df = spark.createDataFrame([], schema=weather_schema)
routes_df = spark.createDataFrame([], schema=route_schema)

# Persist the schemas to Unity Catalog Delta tables (if not exists) for further ETL
catalog = "main"  # Change to your catalog as needed
schema = "cycling" # Schema/Database

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{schema}")

athletes_df.write.format("delta").mode("ignore").saveAsTable(f"{catalog}.{schema}.athletes")
rides_df.write.format("delta").mode("ignore").saveAsTable(f"{catalog}.{schema}.rides")
weather_df.write.format("delta").mode("ignore").saveAsTable(f"{catalog}.{schema}.weather")
routes_df.write.format("delta").mode("ignore").saveAsTable(f"{catalog}.{schema}.routes")

print("Delta schemas for core cycling coach model have been created.")

In [0]:
# Databricks notebook source
# Example Athlete Profile Data ETL
from datetime import date

athlete_profiles = [
    {
        "athlete_id": "strava_athlete_id_1",
        "name": "Alex Smith",
        "birth_date": date(1990, 4, 12),
        "gender": "M",
        "joined_date": date(2022, 1, 1),
        "fitness_level": "Intermediate",
        "training_goal": "Gran Fondo 2024",
        "available_hours_week": 7,
        "injury_history": "None",
        "notes": "Prefers outdoor rides"
    },
    {
        "athlete_id": "strava_athlete_id_2",
        "name": "Maria Lopez",
        "birth_date": date(1985, 7, 23),
        "gender": "F",
        "joined_date": date(2023, 3, 10),
        "fitness_level": "Beginner",
        "training_goal": "Century Ride",
        "available_hours_week": 5,
        "injury_history": "Knee tendonitis in 2022",
        "notes": "Needs acclimatization to heat"
    }
]

# Write athlete profiles to Delta table
athletes_df = spark.createDataFrame(athlete_profiles, schema=athlete_schema)
athletes_df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.athletes")
display(athletes_df)
print("Ingested athlete profiles into Delta table.")

In [0]:
# Databricks notebook source
# Example ETL Logic for Strava, Garmin, TrainingPeaks, Weather
import requests
from pyspark.sql.functions import lit

# Placeholder token retrieval functions for production: use dbutils.secrets.get or similar
STRAVA_TOKEN = "<your_strava_token_here>"
GARMIN_TOKEN = "<your_garmin_token_here>"
TRAININGPEAKS_USER = "<your_trainingpeaks_user>"
TRAININGPEAKS_PW = "<your_trainingpeaks_pw>"
OPENWEATHER_API_KEY = "<your_weather_api_key>"

# Utility: Placeholder function for Strava API ride pull (replace with real call and pagination)
def download_strava_activities(athlete_id: str, since: int):
    headers = {"Authorization": f"Bearer {STRAVA_TOKEN}"}
    url = f"https://www.strava.com/api/v3/athlete/activities?after={since}"  # UNIX timestamp
    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return []
    return response.json()  # List of rides/events

# Utility: Placeholder function for weather API (returns dict per ride)
def get_weather_for_ride(lat, lon, start_time):
    url = f"https://api.openweathermap.org/data/2.5/onecall/timemachine?lat={lat}&lon={lon}&dt={int(start_time.timestamp())}&appid={OPENWEATHER_API_KEY}&units=metric"
    response = requests.get(url)
    if response.status_code != 200:
        return None
    data = response.json()
    weather = data.get("current", {})
    return {
        "temperature_c": weather.get("temp"),
        "humidity_pct": weather.get("humidity"),
        "wind_speed_kmh": weather.get("wind_speed"),
        "wind_direction_deg": weather.get("wind_deg"),
        "precipitation_mm": weather.get("rain", {}).get("1h", 0),
        "visibility_km": weather.get("visibility", 0) / 1000,
        "weather_condition": weather.get("weather", [{}])[0].get("description", "")
    }

# Main ETL scaffold: Fetch activities for a list of athlete_ids, enrich, and append
from datetime import datetime
import uuid

def etl_ingest_rides_for_athletes(athlete_ids, since_days_ago=30):
    all_rides = []
    all_weather = []
    since_ts = int((datetime.now().timestamp()) - since_days_ago * 86400)
    for athlete_id in athlete_ids:
        rides = download_strava_activities(athlete_id, since_ts)
        for ride in rides:
            ride_id = ride.get("id", str(uuid.uuid4()))
            activity_date = datetime.fromisoformat(ride.get("start_date_local", datetime.now().isoformat()))
            route_id = ride.get("route_id", str(uuid.uuid4()))
            # Extract main ride data
            all_rides.append({
                "ride_id": str(ride_id),
                "athlete_id": str(athlete_id),
                "ride_date": activity_date,
                "duration_seconds": ride.get("elapsed_time", None),
                "distance_km": ride.get("distance", 0) / 1000.0,
                "avg_power_watts": ride.get("average_watts"),
                "avg_hr": ride.get("average_heartrate"),
                "max_power_watts": ride.get("max_watts"),
                "max_hr": ride.get("max_heartrate"),
                "ride_type": ride.get("type", "Ride"),
                "weather_id": ride_id,
                "route_id": route_id,
                "perceived_exertion": ride.get("suffer_score", None),
                "notes": ride.get("name", "")
            })
            # Weather enrichment
            if ride.get("start_latitude") and ride.get("start_longitude"):
                weather = get_weather_for_ride(
                    ride["start_latitude"], ride["start_longitude"], activity_date
                )
                if weather:
                    weather["weather_id"] = ride_id
                    weather["ride_id"] = ride_id
                    weather["observation_time"] = activity_date
                    all_weather.append(weather)
    # Write rides to Delta
    if all_rides:
        df_rides = spark.createDataFrame(all_rides, schema=ride_schema)
        df_rides.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.rides")
    # Write weather enrichment to Delta
    if all_weather:
        df_weather = spark.createDataFrame(all_weather, schema=weather_schema)
        df_weather.write.format("delta").mode("append").saveAsTable(f"{catalog}.{schema}.weather")
    print(f"Ingested {len(all_rides)} rides and {len(all_weather)} weather records.")
    return len(all_rides), len(all_weather)

# RUN THE ETL: plug in real athlete IDs, tokens for production
athlete_ids = ["strava_athlete_id_1", "strava_athlete_id_2"]  # Replace with real IDs
etl_ingest_rides_for_athletes(athlete_ids, since_days_ago=30)
